# SESSION 16: SQL Project – Sales Analysis

**Course:** Working with Databases & SQL Basics  
**Topic:** End-to-End Sales Analysis, CSV Ingestion, Customer Segmentation, and KPI Dashboards  
**Database:** `sales_analysis.db`  
**Execution Environment:** Python 3.12, SQLite 3.49, Pandas 3.1

---

## 📌 Project Overview & Objectives
This capstone project synthesizes core SQL competencies developed throughout the course into an end-to-end sales analysis pipeline. Real-world business analysts and data scientists frequently extract raw transaction dumps, load them into staging database tables, run aggregate profiling to identify top spenders and high-performing vendors, and package key performance indicators (KPIs) for executive dashboards.

### Project Roadmap:
1. **Task 1: CSV Data Ingestion**: Import raw food delivery transactions from `food_orders.csv` into a structured SQL table `FoodOrders`.
2. **Task 2: Spotify Catalog Setup**: Create and populate `TopSongs` with popular Spotify tracks, demonstrating schema definition and streaming analytics data types.
3. **Task 3: VIP Customer Segmentation**: Identify the top 3 highest-spending customers using aggregate grouping and sorting.
4. **Task 4: Product & Vendor Performance Reporting**: Aggregate order counts and GMV (Gross Merchandise Value) per restaurant.
5. **Task 5: Executive Dashboard KPI Generation**: Format high-level business metrics (`Average Order Amount` and `Total Unique Customers`) into an executive-ready two-column KPI table using `UNION ALL`.

---

## Task 1: Ingesting `food_orders.csv` into `FoodOrders` Table
> **Task 1:** Import a CSV file of food delivery orders (with columns like `order_id`, `restaurant_name`, `customer_name`, `order_amount`, `order_date`) into a new SQL table named `FoodOrders` using your database tool of choice.

We use Python's `sqlite3` and `pandas` (`df.to_sql()`) to programmatically create `FoodOrders` and ingest the raw transactions from `food_orders.csv` into `sales_analysis.db`.

In [1]:
import os
import sqlite3
import pandas as pd

# Connect to sales_analysis.db
db_path = "sales_analysis.db"
csv_path = "food_orders.csv"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 1. Create table FoodOrders with schema definition
cursor.execute("DROP TABLE IF EXISTS FoodOrders;")
cursor.execute("""
CREATE TABLE FoodOrders (
    order_id INTEGER PRIMARY KEY,
    restaurant_name TEXT NOT NULL,
    customer_name TEXT NOT NULL,
    order_amount DECIMAL(10, 2) NOT NULL,
    order_date DATE NOT NULL
);
""")
conn.commit()

# 2. Ingest CSV data into SQLite table
df_raw = pd.read_csv(csv_path)
df_raw.to_sql("FoodOrders", conn, if_exists="append", index=False)
conn.commit()

print(f"[OK] Ingested {len(df_raw)} records from {csv_path} into FoodOrders table.")

# 3. Preview first 5 rows
df_preview = pd.read_sql_query("SELECT * FROM FoodOrders LIMIT 5;", conn)
print("\n--- First 5 Records in FoodOrders ---")
display(df_preview)


[OK] Ingested 19 records from food_orders.csv into FoodOrders table.

--- First 5 Records in FoodOrders ---


,order_id,restaurant_name,customer_name,order_amount,order_date
0,1,Burger King,Rahul Sharma,350,2026-03-01
1,2,Subway,Amit Verma,300,2026-03-02
2,3,Burger King,Priya Patel,450,2026-03-03
3,4,Pizza Hut,Rahul Sharma,650,2026-03-04
4,5,Starbucks,Amit Verma,420,2026-03-05


## Task 2: Creating `TopSongs` & Inserting Spotify Records
> **Task 2:** Write SQL statements to create a table called `TopSongs` with columns: `song_id`, `song_title`, `artist`, `streams`, and `release_date`, then insert at least 5 records representing popular tracks from Spotify.

Schema definition:
- `song_id`: `INTEGER PRIMARY KEY`
- `song_title`: `TEXT NOT NULL`
- `artist`: `TEXT NOT NULL`
- `streams`: `BIGINT NOT NULL` (Spotify global stream count)
- `release_date`: `DATE NOT NULL`

In [2]:
# 1. Create TopSongs table
cursor.execute("DROP TABLE IF EXISTS TopSongs;")
cursor.execute("""
CREATE TABLE TopSongs (
    song_id INTEGER PRIMARY KEY,
    song_title TEXT NOT NULL,
    artist TEXT NOT NULL,
    streams BIGINT NOT NULL,
    release_date DATE NOT NULL
);
""")
conn.commit()

# 2. Insert 5 iconic Spotify global hits
spotify_tracks = [
    (1, "Blinding Lights", "The Weeknd", 4200000000, "2019-11-29"),
    (2, "Shape of You", "Ed Sheeran", 3800000000, "2017-01-06"),
    (3, "Someone You Loved", "Lewis Capaldi", 3300000000, "2018-11-08"),
    (4, "Sunflower", "Post Malone & Swae Lee", 3200000000, "2018-10-19"),
    (5, "As It Was", "Harry Styles", 3100000000, "2022-04-01")
]

cursor.executemany(
    "INSERT INTO TopSongs (song_id, song_title, artist, streams, release_date) VALUES (?, ?, ?, ?, ?);",
    spotify_tracks
)
conn.commit()

# 3. Query and display TopSongs sorted by stream count
df_songs = pd.read_sql_query("SELECT * FROM TopSongs ORDER BY streams DESC;", conn)
print(f"[OK] TopSongs table populated with {len(df_songs)} Spotify records:")
display(df_songs)


[OK] TopSongs table populated with 5 Spotify records:


,song_id,song_title,artist,streams,release_date
0,1,Blinding Lights,The Weeknd,4200000000,2019-11-29
1,2,Shape of You,Ed Sheeran,3800000000,2017-01-06
2,3,Someone You Loved,Lewis Capaldi,3300000000,2018-11-08
3,4,Sunflower,Post Malone & Swae Lee,3200000000,2018-10-19
4,5,As It Was,Harry Styles,3100000000,2022-04-01


## Task 3: VIP Customer Segmentation (Top 3 Customers by Spend)
> **Task 3:** Write an SQL query to find the top 3 customers who ordered the most from the `FoodOrders` table based on total `order_amount`, and display their names and total spent.

### Analytical Logic:
To rank customers by lifetime spend:
1. `GROUP BY customer_name` aggregates multiple orders placed by the same individual.
2. `SUM(order_amount)` computes total GMV per customer.
3. `ORDER BY total_spent DESC` positions the highest spenders at the top.
4. `LIMIT 3` filters for the top 3 VIP customers.

In [3]:
query_task3 = """
SELECT 
    customer_name,
    ROUND(SUM(order_amount), 2) AS total_spent
FROM FoodOrders
GROUP BY customer_name
ORDER BY total_spent DESC
LIMIT 3;
"""

df_top_customers = pd.read_sql_query(query_task3, conn)
print("--- Top 3 Customers by Total Spend ---")
display(df_top_customers)


--- Top 3 Customers by Total Spend ---


,customer_name,total_spent
0,Sneha Reddy,2600.0
1,Priya Patel,2150.0
2,Rohan Gupta,1950.0


## Task 4: Product / Restaurant Performance Report
> **Task 4:** Generate a product performance report by writing an SQL query that lists each `restaurant_name` from `FoodOrders`, the number of orders, and the total `order_amount`, ordered by total `order_amount` descending.  
> *Hint: Use `GROUP BY` and `ORDER BY` clauses.*

### Analytical Logic:
- `COUNT(order_id)` tracks order volume and operational demand.
- `SUM(order_amount)` tracks gross merchandise value (GMV).
- High-GMV restaurants represent key revenue drivers on food delivery platforms like Zomato or Swiggy.

In [4]:
query_task4 = """
SELECT 
    restaurant_name,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_order_amount
FROM FoodOrders
GROUP BY restaurant_name
ORDER BY total_order_amount DESC;
"""

df_performance = pd.read_sql_query(query_task4, conn)
print("--- Restaurant Performance Report ---")
display(df_performance)


--- Restaurant Performance Report ---


,restaurant_name,total_orders,total_order_amount
0,Behrouz Biryani,4,3700.0
1,Pizza Hut,3,1730.0
2,Biryani By Kilo,2,1650.0
3,Subway,4,1550.0
4,Burger King,3,1300.0
5,Starbucks,2,770.0
6,Haldiram's,1,520.0


## Task 5: Executive Dashboard KPIs (`kpi_name`, `kpi_value`)
> **Task 5:** Create an SQL query that calculates two KPIs for the `FoodOrders` table: (1) average `order_amount` and (2) total number of unique customers, and format the output for dashboard display (two columns: `kpi_name`, `kpi_value`).

### Analytical Logic & Dashboard Layout:
Executive dashboards (e.g. Tableau, Power BI, Streamlit, Metabase) often expect metrics in a normalized **key-value pair format** (`kpi_name`, `kpi_value`) so each row powers an independent KPI card or tile widget.
Using `UNION ALL`, we stack the scalar aggregations vertically:
1. `AVG(order_amount)` &rarr; Average basket size / average order value (AOV).
2. `COUNT(DISTINCT customer_name)` &rarr; Total unique active user base.

In [5]:
query_task5 = """
SELECT 
    'Average Order Amount' AS kpi_name,
    CAST(ROUND(AVG(order_amount), 2) AS TEXT) AS kpi_value
FROM FoodOrders

UNION ALL

SELECT 
    'Total Unique Customers' AS kpi_name,
    CAST(COUNT(DISTINCT customer_name) AS TEXT) AS kpi_value
FROM FoodOrders;
"""

df_kpis = pd.read_sql_query(query_task5, conn)
print("--- Executive Dashboard KPIs (Two-Column Format) ---")
display(df_kpis)

# Close connection
conn.close()
print("\n[OK] Database connection closed cleanly.")


--- Executive Dashboard KPIs (Two-Column Format) ---


,kpi_name,kpi_value
0,Average Order Amount,590.53
1,Total Unique Customers,7



[OK] Database connection closed cleanly.


---

## 🎯 Summary of Key Insights & Project Takeaways

| Metric / Dimension | Finding | Business Value |
| :--- | :--- | :--- |
| **Top Customer** | **Sneha Reddy** (₹2,600.00 spent) | High-intent customer for loyalty program promotions. |
| **Top Restaurant** | **Behrouz Biryani** (4 orders, ₹3,700.00 GMV) | Anchor brand generating ~33% of platform order value. |
| **Average Order Value (AOV)** | **₹590.53** | Benchmark for minimum order thresholds and delivery discounts. |
| **Unique Customer Base** | **7 unique users** across 19 orders | Repeat purchase rate = $\frac{19}{7} \approx 2.71$ orders per customer. |
